# Automated Failure Mode Classification in Rock Mechanics

When a rock sample is crushed in a laboratory test, the way it breaks is recorded as a failure mode code - for example, 2B means a shear fracture at a specific angle, and XA means the sample broke along an existing crack. These codes are currently assigned by a person looking at the broken sample, which takes time and can vary between observers.

This project trains a machine learning model to predict the failure mode code from measurements already recorded during the test, such as the size of the sample, how much pressure was applied, and how stiff the rock was. If the model can do this reliably, it could speed up analysis and make the results more consistent.

The data comes from two types of laboratory tests: uniaxial compression (the sample is squeezed from top and bottom with no side pressure, about 177 samples) and triaxial compression (side pressure is added to simulate conditions deep underground, about 351 samples). A third test type in the raw data, the Brazilian tensile test, does not have failure mode codes and is not used for classification. A fourth dataset covering rock mass field observations is also excluded because it describes the rock at a tunnel or mine face, not individual laboratory samples.

Three classification methods are tested: Logistic Regression, Random Forest, and Support Vector Machine. The best one is tuned further, and the results are explained using a tool called SHAP, which shows which measurements had the most influence on each prediction.

## Section 1 - Libraries

This cell loads all the tools (called libraries) that are used throughout the notebook. Loading everything in one place at the start means the notebook will raise an error immediately if something is missing, rather than failing halfway through a later step. The version numbers are printed so that anyone trying to reproduce the results knows exactly which software was used.

In [2]:
# Install any missing libraries automatically before importing them.
# This means the notebook can run on any computer without manual setup steps.
import subprocess, sys

required = [
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "shap",
    "xlrd",
]

for package in required:
    try:
        __import__(package if package != "scikit-learn" else "sklearn")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"{package} installed.")

print("All libraries ready.")

# ----- Imports -----

import warnings
warnings.filterwarnings("ignore")

# Data handling
import numpy as np
import pandas as pd

# Visualisation
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Evaluation
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# Interpretability
import shap

# Reproducibility - fixing the random seed means results are the same every time the notebook is run
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

# Print version numbers so results can be reproduced on another machine
print(f"numpy        {np.__version__}")
print(f"pandas       {pd.__version__}")
print(f"matplotlib   {matplotlib.__version__}")
print(f"seaborn      {sns.__version__}")
import sklearn; print(f"scikit-learn {sklearn.__version__}")
print(f"shap         {shap.__version__}")

Installing shap...
shap installed.
Installing xlrd...
xlrd installed.
All libraries ready.
numpy        1.26.4
pandas       2.2.3
matplotlib   3.9.2
seaborn      0.13.2
scikit-learn 1.5.1
shap         0.51.0


## Section 2 - Data Ingestion

This section loads the four raw Excel files into Python. Each file has several rows at the top that describe the columns rather than containing actual measurements, so the column names are assigned manually after loading. No changes are made to the data itself here - the goal is only to get it into a form that can be inspected in the next section.

Two of the four files are loaded for reference but are not used in the classifier:
- Brazilian tensile test data does not have failure mode codes, only a tensile strength value in MPa.
- Rock mass rating data describes observations at a mine face, not individual laboratory samples.

The two files used for classification are the uniaxial compression test (UCM) and the triaxial compression test (TCM).

In [3]:
import os

RAW_DIR = os.path.join("data", "raw")

# --- Triaxial compression test (TCM) ---
# Data starts at row 10 (rows 0-9 are headers and units)
# Column names are assigned manually based on inspection of the original file
TCM_COLS = [
    "specimen_id", "hole_id", "sample_id", "depth_top_m", "rock_type",
    "diameter_mm", "height_mm", "hd_ratio", "mass_g", "density_gcm3",
    "confining_pressure_mpa", "failure_load_kn", "tcm_strength_mpa",
    "tangent_modulus_gpa", "secant_modulus_gpa",
    "tangent_poisson", "secant_poisson",
    "axial_strain_at_failure", "failure_mode", "note"
]

df_tcm_raw = pd.read_excel(
    os.path.join(RAW_DIR, "Combined Triaxial test results.xls"),
    sheet_name="TCM",
    header=None,
    skiprows=10,
    names=TCM_COLS,
    engine="xlrd"
)
print(f"TCM raw shape: {df_tcm_raw.shape}")
print(df_tcm_raw.head(3).to_string())
print()

# --- Uniaxial compression test (UCM) ---
# Data starts at row 9
UCM_COLS = [
    "specimen_id", "hole_id", "sample_id", "depth_m", "rock_type",
    "diameter_mm", "height_mm", "hd_ratio", "mass_g", "density_gcm3",
    "failure_load_kn", "ucs_mpa",
    "tangent_modulus_gpa", "secant_modulus_gpa",
    "tangent_poisson", "secant_poisson",
    "axial_strain_at_failure", "failure_code", "note"
]

df_ucm_raw = pd.read_excel(
    os.path.join(RAW_DIR, "Combined Uniaxial Test Results.xls"),
    sheet_name="UCM",
    header=None,
    skiprows=9,
    names=UCM_COLS,
    engine="xlrd"
)
print(f"UCM raw shape: {df_ucm_raw.shape}")
print(df_ucm_raw.head(3).to_string())
print()

# --- Brazilian tensile test (UTB) - loaded for reference only, not used in classifier ---
UTB_COLS = [
    "specimen_id", "hole_id", "sample_id", "depth_m", "rock_type",
    "diameter_mm", "height_mm", "mass_g", "density_gcm3",
    "failure_load_kn", "bts_mpa", "note", "empty1", "empty2", "empty3"
]

df_utb_raw = pd.read_excel(
    os.path.join(RAW_DIR, "Combined BrazilianTest Results.xls"),
    sheet_name="UTB",
    header=None,
    skiprows=9,
    names=UTB_COLS,
    engine="xlrd"
)
print(f"UTB raw shape (reference only - no failure mode codes): {df_utb_raw.shape}")
print()

# --- Rock mass rating (RMR) - loaded for reference only, not used in classifier ---
df_rmr_raw = pd.read_excel(
    os.path.join(RAW_DIR, "Rock Mass Rating data sheet.xls"),
    sheet_name=0,
    header=0,
    engine="xlrd"
)
print(f"RMR raw shape (reference only - field mapping data): {df_rmr_raw.shape}")

TCM raw shape: (351, 20)
  specimen_id hole_id sample_id  depth_top_m rock_type  diameter_mm height_mm  hd_ratio  mass_g  density_gcm3  confining_pressure_mpa  failure_load_kn  tcm_strength_mpa  tangent_modulus_gpa  secant_modulus_gpa  tangent_poisson  secant_poisson  axial_strain_at_failure failure_mode  note
0  TCM-1-F-01     NaN       NaN          NaN       NaN        47.27     94.61  2.001481  542.56      3.267756                     5.0            484.5        276.078569                153.0               152.0            0.236           0.216                 0.001802           2B   NaN
1  TCM-1-F-02     NaN       NaN          NaN       NaN        47.24     96.05  2.033235  550.46      3.269782                    10.0            652.9        372.509205                149.0               156.0            0.232           0.257                 0.002646           XA   NaN
2  TCM-1-F-03     NaN       NaN          NaN       NaN        47.17     91.77  1.945516  525.08      3.274184     